# Website Availability Monitoring — Final Integrated Notebook

**Team members:** Kurbatov Igor, Nikitenko Mikhail, Salkov Ivan, Bassam Ahmed

This notebook integrates the work from the original project files into one clean workflow:

- **Member 1:** website checking and response-time measurement  
- **Member 2:** data storage and summary statistics  
- **Member 3:** visualizations and comparative analysis  
- **Integration:** final structure, outputs, and presentation support

## What this notebook does
1. Imports libraries and defines the monitoring functions  
2. Stores the project website list  
3. Runs monitoring if raw data is not already available  
4. Builds and saves the raw dataset and per-site summary  
5. Merges category information  
6. Produces the project plots and saves them as PNG files  
7. Prints the overall conclusion

> **AI use declaration:** AI assistance was used for restructuring the notebook, improving organization, and preparing a unified final version. The project logic, datasets, and outputs are based on the team’s original files.

In [ ]:
import os
import time
from datetime import datetime

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 120

## 1. Monitoring functions

In [ ]:
def check_website(url, timeout=5):
    """
    Send one HTTP request to a website and return a dictionary with the result.
    """
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    start_time = time.time()

    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers, timeout=timeout)
        end_time = time.time()

        result = {
            "url": url,
            "timestamp": timestamp,
            "status": "success",
            "status_code": response.status_code,
            "response_time": round(end_time - start_time, 3),
            "error_type": None
        }

    except requests.exceptions.Timeout:
        result = {
            "url": url,
            "timestamp": timestamp,
            "status": "failed",
            "status_code": None,
            "response_time": None,
            "error_type": "timeout"
        }

    except requests.exceptions.ConnectionError:
        result = {
            "url": url,
            "timestamp": timestamp,
            "status": "failed",
            "status_code": None,
            "response_time": None,
            "error_type": "connection_error"
        }

    except requests.exceptions.RequestException:
        result = {
            "url": url,
            "timestamp": timestamp,
            "status": "failed",
            "status_code": None,
            "response_time": None,
            "error_type": "request_error"
        }

    return result


def check_all_websites(websites, timeout=5):
    """
    Run the monitoring process for a list of websites.
    """
    results = []
    for url in websites:
        results.append(check_website(url, timeout=timeout))
    return results

## 2. Website list

In [ ]:
# generated list of websites (using AI)
websites = [
    # Foreign websites
    "https://www.google.com",
    "https://github.com",
    "https://www.wikipedia.org",
    "https://stackoverflow.com",
    "https://www.youtube.com",
    "https://web.telegram.org",
    "https://www.cloudflare.com",
    "https://www.reddit.com",
    "https://www.bbc.com",
    "https://www.cnn.com",
    "https://www.nytimes.com",
    "https://www.theguardian.com",
    "https://www.amazon.com",
    "https://www.netflix.com",
    "https://www.apple.com",
    "https://www.microsoft.com",
    "https://www.mozilla.org",
    "https://www.python.org",
    "https://www.openai.com",
    "https://www.imdb.com",
    "https://www.booking.com",
    "https://www.ebay.com",
    "https://www.twitch.tv",
    "https://www.discord.com",
    "https://www.linkedin.com",
    "https://www.dropbox.com",
    "https://www.adobe.com",
    "https://www.medium.com",
    "https://www.quora.com",
    "https://www.paypal.com",

    # Russian websites
    "https://max.ru",
    "https://www.yandex.ru",
    "https://vk.com",
    "https://www.ozon.ru",
    "https://www.wildberries.ru",
    "https://www.avito.ru",
    "https://lenta.ru",
    "https://ria.ru",
    "https://www.sberbank.com",
    "https://www.gosuslugi.ru",
    "https://www.mail.ru",
    "https://ok.ru",
    "https://www.cian.ru",
    "https://www.hh.ru",
    "https://www.rbc.ru",
    "https://www.kommersant.ru",
    "https://www.gazeta.ru",
    "https://www.sport-express.ru",
    "https://www.championat.com",
    "https://www.dns-shop.ru",
    "https://www.mvideo.ru",
    "https://www.citilink.ru",
    "https://www.tinkoff.ru",
    "https://alfabank.ru",
    "https://www.vtb.ru",
    "https://www.consultant.ru",
    "https://www.kremlin.ru",
    "https://www.mos.ru",
    "https://www.rzd.ru",
    "https://www.aeroflot.ru",
    "https://www.sravni.ru"
]

## 3. Load existing data or run the monitoring step

This project already includes the generated CSV files.  
To avoid unnecessary reruns, the notebook first tries to load:

- `monitoring_results_raw.csv`
- `monitoring_results_summary_by_site.csv`

If the raw file is missing, it runs the monitoring step and creates it again.

In [ ]:
RAW_FILE = "monitoring_results_raw.csv"
SUMMARY_FILE = "monitoring_results_summary_by_site.csv"
CATEGORY_FILE = "website_categories.csv"

if os.path.exists(RAW_FILE):
    df_raw = pd.read_csv(RAW_FILE)
    print(f"Loaded existing raw data from {RAW_FILE}")
else:
    print("Raw data file not found. Running the monitoring process...")
    results = check_all_websites(websites)
    df_raw = pd.DataFrame(results)
    df_raw.to_csv(RAW_FILE, index=False)
    print(f"Raw data saved to {RAW_FILE}")

df_raw.head()

## 4. Data storage and summary statistics

In [ ]:
# Ensure a clean working copy
df = df_raw.copy()

# Save raw data again to keep one clear final workflow
df.to_csv(RAW_FILE, index=False)

# Overall summary statistics
total_checks = len(df)
successful_checks = (df["status"] == "success").sum()
failed_checks = total_checks - successful_checks
failure_rate = failed_checks / total_checks if total_checks > 0 else np.nan

response_times = df["response_time"].dropna()
avg_response_time = response_times.mean() if not response_times.empty else np.nan
median_response_time = response_times.median() if not response_times.empty else np.nan
std_response_time = response_times.std() if not response_times.empty else np.nan
min_response_time = response_times.min() if not response_times.empty else np.nan
max_response_time = response_times.max() if not response_times.empty else np.nan

summary_table = pd.DataFrame({
    "metric": [
        "Total checks",
        "Successful checks",
        "Failed checks",
        "Failure rate",
        "Average response time (s)",
        "Median response time (s)",
        "Std. dev. response time (s)",
        "Minimum response time (s)",
        "Maximum response time (s)"
    ],
    "value": [
        total_checks,
        successful_checks,
        failed_checks,
        round(failure_rate, 4),
        round(avg_response_time, 3),
        round(median_response_time, 3),
        round(std_response_time, 3),
        round(min_response_time, 3),
        round(max_response_time, 3)
    ]
})

print("Overall Summary Statistics")
display(summary_table)

# Failure breakdown by error type
failure_breakdown = (
    df[df["status"] == "failed"]
    .groupby("error_type")
    .size()
    .reset_index(name="failure_count")
    .sort_values("failure_count", ascending=False)
)

print("Failure breakdown by error type")
display(failure_breakdown)

# Per-website statistics
per_website_stats = (
    df.groupby("url")
    .agg(
        total_checks=("status", "count"),
        failures=("status", lambda x: (x != "success").sum()),
        failure_rate=("status", lambda x: (x != "success").sum() / len(x) if len(x) else np.nan),
        avg_response_time=("response_time", "mean"),
        min_response_time=("response_time", "min"),
        max_response_time=("response_time", "max"),
    )
    .reset_index()
)

per_website_stats.to_csv(SUMMARY_FILE, index=False)
print(f"Per-website summary saved to {SUMMARY_FILE}")
display(per_website_stats.head(10))

## 5. Merge categories and analysis variables

In [ ]:
helper_df = pd.read_csv(CATEGORY_FILE)
helper_df = helper_df.rename(columns={"URL": "url"})

# Foreign websites = first 30 entries in the original project list
df["foreign"] = df.index < 30

# Blocked/inaccessible websites according to the original project logic
list_of_blocked = [4, 5, 6, 8, 18, 23, 24, 27, 28]
df["blocked"] = df.index.isin(list_of_blocked)

# Merge categories
df = pd.merge(df, helper_df, on="url", how="left")
if "#" in df.columns:
    del df["#"]

# Merge per-site summary statistics
df = pd.merge(df, per_website_stats, on="url", how="left")

df.head()

## 6. Helper functions for analysis

In [ ]:
def calculate_rate(dataset, column="failures"):
    if dataset.empty:
        return 0
    return np.mean(dataset[column])


def get_rates(data, func):
    result = {}
    for key in data:
        result[key] = func(data[key])
    return result

## 7. Plot 1 — Failure rates across groups

In [ ]:
datasets = {
    "full": df,
    "foreign": df[df["foreign"]],
    "foreign except blocked": df[df["foreign"] & (df["blocked"] == False)],
    "blocked": df[df["blocked"]],
    "russian": df[df["foreign"] == False],
}

rates = get_rates(datasets, calculate_rate)

def bar_color(name):
    if "foreign" in name:
        return "gold"
    elif "russian" in name:
        return "red"
    elif "blocked" in name:
        return "green"
    return "blue"

colors = [bar_color(name) for name in rates]

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(rates.keys(), rates.values(), color=colors)
ax.set_title("Failure rates across different website groups")
ax.set_ylabel("Failure rate")
ax.set_ylim(0, 1)
plt.xticks(rotation=15)
plt.tight_layout()
fig.savefig("failure_rates_by_group.png", dpi=300, bbox_inches="tight")
plt.show()

## 8. Plot 2 — Failure and success rates by category

In [ ]:
category_dict = {}
for category, group in df.groupby("Category"):
    category_dict[category] = get_rates(
        {("foreign" if foreign else "russian"): values for foreign, values in group.groupby("foreign")},
        calculate_rate
    )

cats = list(category_dict.keys())
foreign_fail = [category_dict[cat].get("foreign", 0) for cat in cats]
russian_fail = [category_dict[cat].get("russian", 0) for cat in cats]
foreign_success = [1 - x for x in foreign_fail]
russian_success = [1 - x for x in russian_fail]

x = np.arange(len(cats))
width = 0.3

fig, ax = plt.subplots(figsize=(11, 7))
ax.bar(x - 0.05 - width / 2, foreign_fail, width, label="Foreign fail", color="lightcoral")
ax.bar(x - 0.05 - width / 2, foreign_success, width, bottom=foreign_fail, label="Foreign success", color="lightgreen")
ax.bar(x + width / 2, russian_fail, width, label="Russian fail", color="red")
ax.bar(x + width / 2, russian_success, width, bottom=russian_fail, label="Russian success", color="green")

ax.set_xticks(x)
ax.set_xticklabels(cats, rotation=45, ha="right")
ax.set_ylabel("Proportion")
ax.set_title("Failure and success rates by category")
ax.legend()
plt.tight_layout()
fig.savefig("failure_rate_by_category.png", dpi=300, bbox_inches="tight")
plt.show()

## 9. Plot 3 — Blocked websites by category

In [ ]:
blocked_by_category = {category: data for category, data in df[df["blocked"]].groupby("Category")}
vals = get_rates(blocked_by_category, calculate_rate)

fail = list(vals.values())
success = [1 - x for x in fail]
bottom = np.zeros(len(fail))

fig, ax = plt.subplots(figsize=(9, 6))
for label, values in [("Fail", fail), ("Success", success)]:
    ax.bar(vals.keys(), values, bottom=bottom, label=label, color=("red" if label == "Fail" else "green"))
    bottom += values

ax.set_title("Failure rates for blocked websites by category")
ax.set_ylabel("Proportion")
ax.legend()
plt.xticks(rotation=15)
plt.tight_layout()
fig.savefig("blocked_sites_category_counts.png", dpi=300, bbox_inches="tight")
plt.show()

## 10. Plot 4 — Response time comparison

In [ ]:
response_times_ru = df.loc[df["foreign"] == False, "response_time"].dropna()
response_times_f = df.loc[df["foreign"], "response_time"].dropna()

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(np.arange(len(response_times_f)), response_times_f, color="red", label="foreign")
ax.scatter(np.arange(len(response_times_ru)), response_times_ru, color="blue", label="russian")
ax.set_title("Distribution of response times")
ax.set_ylabel("Response time (seconds)")
ax.legend(loc=2)
plt.tight_layout()
fig.savefig("response_time_boxplot.png", dpi=300, bbox_inches="tight")
plt.show()

## 11. Final conclusion

In [ ]:
def conclusion(dataframe):
    connectivity_f = 1 - calculate_rate(dataframe[dataframe["foreign"]])
    connectivity_ru = 1 - calculate_rate(dataframe[dataframe["foreign"] == False])

    if connectivity_f >= 0.8 and connectivity_ru >= 0.8:
        return "No problems with connection"
    elif connectivity_f >= 0.8:
        return "No problems with foreign connections"
    elif connectivity_f <= 0.3 and connectivity_ru <= 0.3:
        return "Very weak connection"
    elif connectivity_ru >= 0.6:
        return "Trouble with foreign connections"
    else:
        return "There are some problems with the connection"

print("Overall conclusion:", conclusion(df))

## 12. Output files produced by this notebook

Running this notebook creates or updates the following files:

- `monitoring_results_raw.csv`
- `monitoring_results_summary_by_site.csv`
- `failure_rates_by_group.png`
- `failure_rate_by_category.png`
- `blocked_sites_category_counts.png`
- `response_time_boxplot.png`

This gives you one complete notebook for the full project workflow.